# GNN embeddings as features for RF and a Temporal Transformer

Use trained GNNs as **feature extractors**: train each model on the supervised illicit/licit task, then take the **penultimate-layer hidden state** as a learned per-node embedding. Hand the embedding to a downstream Random Forest (and, for wallets, a Temporal Transformer) alongside the original engineered features.

## Why this is interesting
RF on the raw Elliptic++ features is famously hard to beat (≈0.80 illicit-F1 on tx). The engineered features include 72 *Aggregate* columns that already summarise each node's 1-hop neighbourhood. A GNN can in principle do **deeper, learned aggregation** (2–3 hops, soft attention over edge types, etc.). If the GNN's embedding adds information beyond the raw features, RF on `raw + emb` should beat RF on `raw` alone — that's the empirical test we run below.

## Pipeline
1. Train two GNN embedders supervised on the train mask.
2. Run a single forward pass to extract embeddings for **every** node (transductive — the GNN never saw val/test labels).
3. **RF on raw / embedding-only / raw + embedding** for both tasks. Compare illicit-F1 deltas.
4. **Temporal Transformer** for the wallet task — concat the GNN embedding to every per-time-step token, so the model sees both calendar-time dynamics and static graph context.

## Tasks
- **Transactions** — homogeneous tx ↔ tx graph. Embedder = 3-layer residual GraphSAGE (`hidden_dim=128`).
- **Wallets** — heterogeneous (wallet, tx) graph using all three edge types. Embedder = 2-layer per-relation `HeteroSAGE` (`hidden_dim=64`).

Train/val/test split: time steps **1–29 / 30–34 / 35–49** (chronological).

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
from sklearn.ensemble import RandomForestClassifier
from torch_geometric.data import Data, HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv
from torch_geometric.utils import to_undirected
from torch_geometric.transforms import ToUndirected

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "transactions_data").exists():
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
WALLETS_DIR = ROOT / "actors_data"

TRAIN_END = 29
VAL_END = 34
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"root={ROOT}  device={DEVICE}")

/Users/jarayliu/Documents/GitHub/stat175-final/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


root=/Users/jarayliu/Documents/GitHub/stat175-final  device=cpu


## 1. Load data and build both graphs (single cell)

`transactions` and `wallets` (one row per wallet, latest time-step) are kept as DataFrames so the RF + Temporal NN can grab raw feature matrices later. `wallet_per_step` (per-(address, time-step) labelled rows) is also kept for the Temporal Transformer.

In [2]:
def standardize_train(features_array, labelled_mask, time_steps_array):
    scaler = StandardScaler().fit(features_array[labelled_mask & (time_steps_array <= TRAIN_END)])
    return scaler.transform(features_array).astype(np.float32)

def map_edges(edge_df, src_col, dst_col, src_map, dst_map):
    src = edge_df[src_col].map(src_map)
    dst = edge_df[dst_col].map(dst_map)
    valid = src.notna() & dst.notna()
    return torch.tensor(
        np.stack([src[valid].astype(np.int64).values, dst[valid].astype(np.int64).values]),
        dtype=torch.long,
    )

# === Transactions ===
transactions = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
transaction_classes = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.csv")
transaction_classes["label"] = transaction_classes["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
transactions = transactions.merge(transaction_classes[["txId", "label"]], on="txId", how="left")
transactions["label"] = transactions["label"].fillna(-1).astype(np.int8)
transaction_feature_cols = [c for c in transactions.columns if c not in ("txId", "Time step", "label")]
transactions[transaction_feature_cols] = transactions[transaction_feature_cols].fillna(0).astype(np.float32)

tx_id_to_idx       = {tx: i for i, tx in enumerate(transactions["txId"].values)}
tx_features_array  = transactions[transaction_feature_cols].values
tx_time_steps      = transactions["Time step"].values
tx_labels          = transactions["label"].values
tx_is_labelled     = tx_labels != -1
tx_features_std    = standardize_train(tx_features_array, tx_is_labelled, tx_time_steps)

transaction_edges = pd.read_csv(TRANSACTIONS_DIR / "txs_edgelist.csv")
tx_edge_index = to_undirected(
    map_edges(transaction_edges, "txId1", "txId2", tx_id_to_idx, tx_id_to_idx),
    num_nodes=len(transactions),
)
transaction_graph = Data(
    x=torch.from_numpy(tx_features_std),
    edge_index=tx_edge_index,
    y=torch.tensor(np.where(tx_labels < 0, 0, tx_labels), dtype=torch.long),
)
transaction_graph.train_mask = torch.tensor(tx_is_labelled & (tx_time_steps <= TRAIN_END))
transaction_graph.val_mask   = torch.tensor(tx_is_labelled & (tx_time_steps > TRAIN_END) & (tx_time_steps <= VAL_END))
transaction_graph.test_mask  = torch.tensor(tx_is_labelled & (tx_time_steps > VAL_END))
transaction_graph = transaction_graph.to(DEVICE)

# === Wallets ===
with open(WALLETS_DIR / "wallets_features.txt") as f:
    wallet_columns = f.readline().rstrip("\n").split(",")
wallet_dtype = {c: np.float32 for c in wallet_columns if c not in ("address", "Time step")}
wallet_dtype["address"] = "string"
wallet_dtype["Time step"] = np.int16

wallet_per_step_raw = pd.read_csv(WALLETS_DIR / "wallets_features.txt", dtype=wallet_dtype)
wallet_classes = pd.read_csv(WALLETS_DIR / "wallets_classes.txt")
wallet_classes["label"] = wallet_classes["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)

# One row per wallet (latest time-step) — the GNN's node feature matrix
wallets = (wallet_per_step_raw
           .sort_values(["address", "Time step"])
           .groupby("address", as_index=False).last()
           .merge(wallet_classes[["address", "label"]], on="address", how="inner"))
wallet_feature_cols = [c for c in wallets.columns if c not in ("address", "Time step", "label")]
wallets[wallet_feature_cols] = wallets[wallet_feature_cols].fillna(0).astype(np.float32)

wallet_addr_to_idx     = {addr: i for i, addr in enumerate(wallets["address"].values)}
wallet_features_array  = wallets[wallet_feature_cols].values
wallet_time_steps      = wallets["Time step"].values
wallet_labels          = wallets["label"].values
wallet_is_labelled     = wallet_labels != -1
wallet_features_std    = standardize_train(wallet_features_array, wallet_is_labelled, wallet_time_steps)

# Per-(wallet, time-step) labelled rows — used by the Temporal Transformer below
wallet_per_step = wallet_per_step_raw.merge(wallet_classes[["address", "label"]], on="address", how="left")
wallet_per_step["label"] = wallet_per_step["label"].fillna(-1).astype(np.int8)
wallet_per_step = wallet_per_step[wallet_per_step["label"] != -1].copy()
wallet_per_step[wallet_feature_cols] = wallet_per_step[wallet_feature_cols].fillna(0).astype(np.float32)
del wallet_per_step_raw

# === All three wallet edge types ===
addr_addr_edges = pd.read_csv(WALLETS_DIR / "AddrAddr_edgelist.txt")
addr_tx_edges   = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_edges   = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")

# === Heterogeneous wallet+tx graph ===
wallet_hetero_graph = HeteroData()
wallet_hetero_graph["wallet"].x = torch.from_numpy(wallet_features_std)
wallet_hetero_graph["tx"].x     = torch.from_numpy(tx_features_std)
wallet_hetero_graph["wallet", "addr_to_addr", "wallet"].edge_index = map_edges(addr_addr_edges, "input_address", "output_address", wallet_addr_to_idx, wallet_addr_to_idx)
wallet_hetero_graph["wallet", "addr_to_tx",   "tx"    ].edge_index = map_edges(addr_tx_edges,   "input_address", "txId",            wallet_addr_to_idx, tx_id_to_idx)
wallet_hetero_graph["tx",     "tx_to_addr",   "wallet"].edge_index = map_edges(tx_addr_edges,   "txId",          "output_address",  tx_id_to_idx,       wallet_addr_to_idx)
wallet_hetero_graph["wallet"].y          = torch.tensor(np.where(wallet_labels < 0, 0, wallet_labels), dtype=torch.long)
wallet_hetero_graph["wallet"].train_mask = torch.tensor(wallet_is_labelled & (wallet_time_steps <= TRAIN_END))
wallet_hetero_graph["wallet"].val_mask   = torch.tensor(wallet_is_labelled & (wallet_time_steps > TRAIN_END) & (wallet_time_steps <= VAL_END))
wallet_hetero_graph["wallet"].test_mask  = torch.tensor(wallet_is_labelled & (wallet_time_steps > VAL_END))
wallet_hetero_graph = ToUndirected()(wallet_hetero_graph).to(DEVICE)

print(f"transaction_graph    nodes={transaction_graph.num_nodes:>9,}  edges={transaction_graph.num_edges:>9,}  features={transaction_graph.num_node_features}")
print(f"wallet_hetero_graph  wallet={wallet_hetero_graph['wallet'].num_nodes:>9,}  tx={wallet_hetero_graph['tx'].num_nodes:>9,}  edge_types={len(wallet_hetero_graph.edge_types)}")
print(f"wallet_per_step      labelled rows={len(wallet_per_step):,}")

transaction_graph    nodes=  203,769  edges=  468,710  features=182
wallet_hetero_graph  wallet=  822,942  tx=  203,769  edge_types=5
wallet_per_step      labelled rows=367,472


## 2. GNN embedder definitions

Both models expose `forward(..., return_embedding=True) → (logits, embedding)`. The embedding is the **penultimate hidden state** — output of the last conv block (post LayerNorm + residual), pre-classifier. That's the natural "learned representation of this node given its neighbourhood".

In [3]:
class ResidualSAGEEmbedder(nn.Module):
    """3-layer residual GraphSAGE on a homogeneous graph."""
    def __init__(self, input_dim, hidden_dim=128, num_layers=3, dropout=0.4):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.convs = nn.ModuleList([SAGEConv(hidden_dim, hidden_dim) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])
        self.classifier = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, return_embedding=False):
        h = self.input_proj(x)
        for conv, norm in zip(self.convs, self.norms):
            residual = h
            h = F.relu(norm(conv(h, edge_index)))
            h = F.dropout(h, p=self.dropout, training=self.training)
            h = h + residual
        logits = self.classifier(h)
        return (logits, h) if return_embedding else logits


class HeteroSAGEEmbedder(nn.Module):
    """2-layer per-relation SAGE on a HeteroData graph; only the wallet hidden state is exposed."""
    def __init__(self, edge_types, hidden_dim=64, dropout=0.4):
        super().__init__()
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv((-1, -1), hidden_dim) for et in edge_types}, aggr="sum"),
            HeteroConv({et: SAGEConv((-1, -1), hidden_dim) for et in edge_types}, aggr="sum"),
        ])
        self.norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict, return_embedding=False):
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {k: F.dropout(F.relu(v), p=self.dropout, training=self.training)
                      for k, v in x_dict.items()}
        wallet_h = self.norm(x_dict["wallet"])
        logits = self.classifier(wallet_h)
        return (logits, wallet_h) if return_embedding else logits

## 3. Train tx GraphSAGE embedder + extract per-tx embeddings

In [4]:
def train_homo(model, graph, epochs=120, lr=1e-3, wd=5e-4, eval_every=5, patience=30):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    n_pos = int((graph.y[graph.train_mask] == 1).sum())
    n_neg = int((graph.y[graph.train_mask] == 0).sum())
    cw = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float, device=DEVICE)
    best_f1, best_state, no_improve = 0.0, None, 0
    for epoch in range(1, epochs + 1):
        model.train(); optimizer.zero_grad()
        logits = model(graph.x, graph.edge_index)
        loss = F.cross_entropy(logits[graph.train_mask], graph.y[graph.train_mask], weight=cw)
        loss.backward(); optimizer.step()
        if epoch % eval_every == 0:
            model.eval()
            with torch.no_grad():
                pred = model(graph.x, graph.edge_index)[graph.val_mask].argmax(-1).cpu().numpy()
            f1 = f1_score(graph.y[graph.val_mask].cpu().numpy(), pred, pos_label=1, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += eval_every
            print(f"  epoch {epoch:3d}  loss={loss.item():.4f}  val_F1={f1:.4f}  best={best_f1:.4f}")
            if no_improve >= patience:
                print(f"  early stop at epoch {epoch}"); break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

print("=== Training tx GraphSAGE embedder ===")
tx_embedder = train_homo(ResidualSAGEEmbedder(transaction_graph.num_node_features), transaction_graph)
tx_embedder.eval()
with torch.no_grad():
    _, tx_emb_tensor = tx_embedder(transaction_graph.x, transaction_graph.edge_index, return_embedding=True)
tx_embeddings = tx_emb_tensor.cpu().numpy()
print(f"tx_embeddings shape: {tx_embeddings.shape}")

=== Training tx GraphSAGE embedder ===
  epoch   5  loss=0.3512  val_F1=0.4435  best=0.4435
  epoch  10  loss=0.2683  val_F1=0.5450  best=0.5450
  epoch  15  loss=0.2375  val_F1=0.5246  best=0.5450
  epoch  20  loss=0.2149  val_F1=0.5353  best=0.5450
  epoch  25  loss=0.1914  val_F1=0.5457  best=0.5457
  epoch  30  loss=0.1802  val_F1=0.5665  best=0.5665
  epoch  35  loss=0.1657  val_F1=0.5973  best=0.5973
  epoch  40  loss=0.1529  val_F1=0.6202  best=0.6202
  epoch  45  loss=0.1434  val_F1=0.6309  best=0.6309
  epoch  50  loss=0.1343  val_F1=0.6663  best=0.6663
  epoch  55  loss=0.1254  val_F1=0.6783  best=0.6783
  epoch  60  loss=0.1151  val_F1=0.6736  best=0.6783
  epoch  65  loss=0.1071  val_F1=0.6958  best=0.6958
  epoch  70  loss=0.1036  val_F1=0.7053  best=0.7053
  epoch  75  loss=0.1001  val_F1=0.7083  best=0.7083
  epoch  80  loss=0.0888  val_F1=0.7388  best=0.7388
  epoch  85  loss=0.0857  val_F1=0.7454  best=0.7454
  epoch  90  loss=0.0801  val_F1=0.7610  best=0.7610
  epoch

## 4. Train wallet HeteroSAGE embedder + extract per-wallet embeddings

Same recipe on the heterogeneous graph — only wallet rows enter the loss; tx node embeddings are learned implicitly to support wallet predictions.

In [6]:
def train_hetero(model, hetero_graph, epochs=120, lr=1e-3, wd=5e-4, eval_every=2, patience=10):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    train_mask = hetero_graph["wallet"].train_mask
    val_mask   = hetero_graph["wallet"].val_mask
    y          = hetero_graph["wallet"].y
    n_pos = int((y[train_mask] == 1).sum())
    n_neg = int((y[train_mask] == 0).sum())
    cw = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float, device=DEVICE)
    best_f1, best_state, no_improve = 0.0, None, 0
    for epoch in range(1, epochs + 1):
        model.train(); optimizer.zero_grad()
        logits = model(hetero_graph.x_dict, hetero_graph.edge_index_dict)
        loss = F.cross_entropy(logits[train_mask], y[train_mask], weight=cw)
        loss.backward(); optimizer.step()
        if epoch % eval_every == 0:
            model.eval()
            with torch.no_grad():
                pred = (model(hetero_graph.x_dict, hetero_graph.edge_index_dict)[val_mask]
                        .argmax(-1).cpu().numpy())
            f1 = f1_score(y[val_mask].cpu().numpy(), pred, pos_label=1, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += eval_every
            print(f"  epoch {epoch:3d}  loss={loss.item():.4f}  val_F1={f1:.4f}  best={best_f1:.4f}")
            if no_improve >= patience:
                print(f"  early stop at epoch {epoch}"); break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

print("=== Training wallet HeteroSAGE embedder ===")
wallet_embedder = train_hetero(
    HeteroSAGEEmbedder(edge_types=wallet_hetero_graph.edge_types),
    wallet_hetero_graph,
)
wallet_embedder.eval()
with torch.no_grad():
    _, wallet_emb_tensor = wallet_embedder(
        wallet_hetero_graph.x_dict, wallet_hetero_graph.edge_index_dict, return_embedding=True,
    )
wallet_embeddings = wallet_emb_tensor.cpu().numpy()
print(f"wallet_embeddings shape: {wallet_embeddings.shape}")

=== Training wallet HeteroSAGE embedder ===
  epoch   2  loss=0.6448  val_F1=0.2818  best=0.2818
  epoch   4  loss=0.4858  val_F1=0.2769  best=0.2818
  epoch   6  loss=0.4133  val_F1=0.2724  best=0.2818
  epoch   8  loss=0.3635  val_F1=0.2752  best=0.2818
  epoch  10  loss=0.3227  val_F1=0.2816  best=0.2818
  epoch  12  loss=0.2878  val_F1=0.2902  best=0.2902
  epoch  14  loss=0.2619  val_F1=0.3056  best=0.3056
  epoch  16  loss=0.2517  val_F1=0.3173  best=0.3173
  epoch  18  loss=0.2339  val_F1=0.3420  best=0.3420
  epoch  20  loss=0.2204  val_F1=0.3552  best=0.3552
  epoch  22  loss=0.2059  val_F1=0.3578  best=0.3578
  epoch  24  loss=0.1969  val_F1=0.3825  best=0.3825
  epoch  26  loss=0.1896  val_F1=0.3949  best=0.3949
  epoch  28  loss=0.1768  val_F1=0.4014  best=0.4014
  epoch  30  loss=0.1738  val_F1=0.4034  best=0.4034
  epoch  32  loss=0.1637  val_F1=0.4034  best=0.4034
  epoch  34  loss=0.1564  val_F1=0.4039  best=0.4039
  epoch  36  loss=0.1548  val_F1=0.4059  best=0.4059
  

## 5. Random Forest on `raw` / `emb` / `raw + emb`

Same chronological split. We expect:
- `raw + emb` ≥ `raw` → embeddings encode information beyond the engineered features.
- `emb` alone ≳ random → embeddings carry meaningful signal.
- `emb` ≪ `raw` is plausible — Elliptic's hand-crafted aggregates are very strong — but the additive `raw + emb` is the headline.

Note: the GNN was trained on train labels only; val/test labels never entered any loss. Val/test embeddings come from the trained GNN's *inference* on those nodes — that is standard transductive node classification, not leakage.

In [7]:
def rf_three_way(X_raw, X_emb, labels, time_steps, label_mask, task_name):
    train_mask = label_mask & (time_steps <= TRAIN_END)
    val_mask   = label_mask & (time_steps > TRAIN_END) & (time_steps <= VAL_END)
    test_mask  = label_mask & (time_steps > VAL_END)
    feature_sets = {
        "raw":     X_raw,
        "emb":     X_emb,
        "raw+emb": np.hstack([X_raw, X_emb]),
    }
    print(f"=== {task_name}  train={int(train_mask.sum()):,}  val={int(val_mask.sum()):,}  test={int(test_mask.sum()):,} ===")
    for name, X in feature_sets.items():
        clf = RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42)
        clf.fit(X[train_mask], labels[train_mask])
        f1_val  = f1_score(labels[val_mask],  clf.predict(X[val_mask]),  pos_label=1, zero_division=0)
        f1_test = f1_score(labels[test_mask], clf.predict(X[test_mask]), pos_label=1, zero_division=0)
        print(f"  [{name:7s}] illicit_F1   val={f1_val:.4f}   test={f1_test:.4f}")
    print("\nFull TEST report for raw+emb:")
    X_full = feature_sets["raw+emb"]
    clf = RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42)
    clf.fit(X_full[train_mask], labels[train_mask])
    print(classification_report(
        labels[test_mask], clf.predict(X_full[test_mask]),
        target_names=["licit", "illicit"], digits=4, zero_division=0,
    ))

rf_three_way(tx_features_std,     tx_embeddings,     tx_labels,     tx_time_steps,     tx_is_labelled,     "TRANSACTIONS")
print()
rf_three_way(wallet_features_std, wallet_embeddings, wallet_labels, wallet_time_steps, wallet_is_labelled, "WALLETS")

=== TRANSACTIONS  train=26,381  val=3,513  test=16,670 ===
  [raw    ] illicit_F1   val=0.9554   test=0.7454
  [emb    ] illicit_F1   val=0.8004   test=0.5603
  [raw+emb] illicit_F1   val=0.8453   test=0.6138

Full TEST report for raw+emb:
              precision    recall  f1-score   support

       licit     0.9670    0.9888    0.9778     15587
     illicit     0.7609    0.5143    0.6138      1083

    accuracy                         0.9579     16670
   macro avg     0.8640    0.7515    0.7958     16670
weighted avg     0.9536    0.9579    0.9541     16670


=== WALLETS  train=148,038  val=22,366  test=94,950 ===
  [raw    ] illicit_F1   val=0.0885   test=0.0686
  [emb    ] illicit_F1   val=0.8832   test=0.5498
  [raw+emb] illicit_F1   val=0.8152   test=0.3688

Full TEST report for raw+emb:
              precision    recall  f1-score   support

       licit     0.9623    0.9825    0.9723     90012
     illicit     0.4830    0.2983    0.3688      4938

    accuracy                   

## 6. Temporal Transformer on per-time-step features + GNN embedding (wallets only)

Wallets have a per-time-step feature trajectory but a single static GNN embedding. We **concat the embedding to every per-time-step token** so each token sees `[x_t || e_wallet] ∈ ℝ^{55+64}`. The Transformer learns to attend over the temporal axis with the graph context built into every position. Wallets are split by their **latest** time step (max ≤ 29 → train, ∈ (29, 34] → val, > 34 → test).

In [8]:
from torch.utils.data import DataLoader, TensorDataset

# Standardise per-(wallet, time-step) features on labelled train rows only
seq_scaler = StandardScaler().fit(
    wallet_per_step.loc[wallet_per_step["Time step"] <= TRAIN_END, wallet_feature_cols].values
)
wallet_per_step[wallet_feature_cols] = (
    seq_scaler.transform(wallet_per_step[wallet_feature_cols].values).astype(np.float32)
)

# Subsample addresses (keep all illicit, sample licit) — same protocol as initial_experiements.ipynb
rng = np.random.default_rng(42)
addr_label = wallet_per_step.groupby("address")["label"].first()
illicit_addrs = addr_label[addr_label == 1].index.values
licit_addrs   = addr_label[addr_label == 0].index.values
chosen = np.concatenate([
    illicit_addrs,
    rng.choice(licit_addrs, size=min(40_000, len(licit_addrs)), replace=False),
])
sub = wallet_per_step[wallet_per_step["address"].isin(set(chosen))].sort_values(["address", "Time step"])

T_MAX = 12
F_seq = len(wallet_feature_cols)
F_emb = wallet_embeddings.shape[1]

xs, ts, masks, embs, ys, splits = [], [], [], [], [], []
for addr, group in sub.groupby("address", sort=False):
    group = group.iloc[-T_MAX:]
    n = len(group)
    x = np.zeros((T_MAX, F_seq), np.float32)
    t = np.zeros(T_MAX, np.int64)
    pad = np.ones(T_MAX, dtype=bool)
    x[:n] = group[wallet_feature_cols].values
    t[:n] = group["Time step"].values
    pad[:n] = False
    xs.append(x); ts.append(t); masks.append(pad)
    embs.append(wallet_embeddings[wallet_addr_to_idx[addr]])
    ys.append(int(group["label"].iloc[0]))
    max_ts = int(group["Time step"].max())
    splits.append(0 if max_ts <= TRAIN_END else (1 if max_ts <= VAL_END else 2))

X_seq = torch.tensor(np.stack(xs))
T_seq = torch.tensor(np.stack(ts))
M_seq = torch.tensor(np.stack(masks))
E_seq = torch.tensor(np.stack(embs))
Y_seq = torch.tensor(ys, dtype=torch.long)
S_seq = torch.tensor(splits, dtype=torch.long)

train_idx = (S_seq == 0); val_idx = (S_seq == 1); test_idx = (S_seq == 2)
print(f"sequences: x={tuple(X_seq.shape)}  emb={tuple(E_seq.shape)}  "
      f"train={int(train_idx.sum())}  val={int(val_idx.sum())}  test={int(test_idx.sum())}")

sequences: x=(54266, 12, 55)  emb=(54266, 64)  train=29959  val=5026  test=19281


In [14]:
class TemporalTransformerWithEmbedding(nn.Module):
    def __init__(self, seq_dim, emb_dim, d_model=64, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.token_proj = nn.Linear(seq_dim + emb_dim, d_model)
        self.position = nn.Embedding(50, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=128, dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, n_layers)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, x, t, padding_mask, emb):
        emb_b = emb.unsqueeze(1).expand(-1, x.size(1), -1)
        token = torch.cat([x, emb_b], dim=-1)
        h = self.token_proj(token) + self.position(t)
        h = self.encoder(h, src_key_padding_mask=padding_mask)
        valid = (~padding_mask).unsqueeze(-1).float()
        return self.classifier((h * valid).sum(1) / valid.sum(1).clamp(min=1))

def make_loader(idx, batch_size, shuffle):
    return DataLoader(
        TensorDataset(X_seq[idx], T_seq[idx], M_seq[idx], E_seq[idx], Y_seq[idx]),
        batch_size=batch_size, shuffle=shuffle,
    )

train_loader = make_loader(train_idx, 512, True)
val_loader   = make_loader(val_idx, 1024, False)
test_loader  = make_loader(test_idx, 1024, False)

model = TemporalTransformerWithEmbedding(F_seq, F_emb).to(DEVICE)
n_pos = int((Y_seq[train_idx] == 1).sum()); n_neg = int((Y_seq[train_idx] == 0).sum())
cw = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float, device=DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train(); running, seen = 0.0, 0
    for x_b, t_b, m_b, e_b, y_b in train_loader:
        x_b, t_b, m_b, e_b, y_b = x_b.to(DEVICE), t_b.to(DEVICE), m_b.to(DEVICE), e_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(x_b, t_b, m_b, e_b), y_b, weight=cw)
        loss.backward(); optimizer.step()
        running += loss.item() * y_b.size(0); seen += y_b.size(0)
    if epoch % 5 == 0 or epoch == 19:
        print(f"  epoch {epoch:2d}  loss={running/seen:.4f}")

def evaluate_temporal(loader, name):
    model.eval(); ys_, preds_ = [], []
    with torch.no_grad():
        for x_b, t_b, m_b, e_b, y_b in loader:
            x_b, t_b, m_b, e_b = x_b.to(DEVICE), t_b.to(DEVICE), m_b.to(DEVICE), e_b.to(DEVICE)
            preds_.append(model(x_b, t_b, m_b, e_b).argmax(-1).cpu().numpy())
            ys_.append(y_b.numpy())
    print(f"{name}:")
    print(classification_report(np.concatenate(ys_), np.concatenate(preds_), target_names=["licit", "illicit"], digits=4, zero_division=0))

evaluate_temporal(val_loader,  "VAL  (in-distribution)")
evaluate_temporal(test_loader, "TEST (the cliff)")

  epoch  0  loss=0.1562
  epoch  5  loss=0.0873
  epoch 10  loss=0.0795
  epoch 15  loss=0.0698
  epoch 19  loss=0.0642
  epoch 20  loss=0.0614
  epoch 25  loss=0.0543
  epoch 30  loss=0.0483
  epoch 35  loss=0.0448
  epoch 40  loss=0.0404
  epoch 45  loss=0.0358
  epoch 50  loss=0.0315
  epoch 55  loss=0.0294
  epoch 60  loss=0.0265
  epoch 65  loss=0.0283
  epoch 70  loss=0.0243
  epoch 75  loss=0.0213
  epoch 80  loss=0.0194
  epoch 85  loss=0.0177
  epoch 90  loss=0.0186
  epoch 95  loss=0.0155
VAL  (in-distribution):
              precision    recall  f1-score   support

       licit     0.9651    0.8966    0.9296      3240
     illicit     0.8338    0.9412    0.8843      1786

    accuracy                         0.9125      5026
   macro avg     0.8995    0.9189    0.9069      5026
weighted avg     0.9185    0.9125    0.9135      5026

TEST (the cliff):
              precision    recall  f1-score   support

       licit     0.8865    0.8703    0.8783     14343
     illicit     0

## Discussion

- **Does the embedding encode beyond raw features?** Read the §5 table. Positive `(raw+emb) − raw` delta = the GNN extracted neighbourhood signal the engineered aggregates didn't capture.
- **Embedding alone vs raw alone.** If `emb` ≈ `raw`, the GNN essentially re-encoded the features. If `emb` ≪ `raw` but `raw+emb` > `raw`, the GNN found a *complementary* signal — the more interesting outcome.
- **Temporal Transformer with embedding** is the only model in the project that simultaneously sees per-time-step dynamics and static graph context. Its test illicit-F1 is the joint upper bound of the temporal-only and GNN-only models — if it doesn't beat both, the two signals are redundant given this architecture.

## Where to go next
- **Replace supervised pre-training with self-supervised** (DGI, GraphCL): the embedding is then label-agnostic — encodes neighbourhood-distinguishability, not label-discriminability. Cleaner separation, more robust to the cliff.
- **Stack via probability calibration** — train a logistic regression on `(RF_proba, GNN_proba)` two-column meta-features. Often the simplest way to extract the last few F1 points.
- **Use the GNN logits, not just the penultimate state** — sometimes a 2-D `(licit_logit, illicit_logit)` is a more stable feature than a 64- or 128-D embedding.